In [ ]:
# --------------------------------------------------
# БЛОК 1: Загрузка данных и дообучение ViT
# --------------------------------------------------
import os
import re
import requests
import zipfile
from collections import defaultdict
from PIL import Image
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

# Настройки
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

TRAIN_PQ = "train_dataset.parquet"
TEST_PQ = "test_dataset.parquet"
TRAIN_IMG_ZIP = "train_images.zip"
TEST_IMG_ZIP = "test_images.zip"
TRAIN_IMG_DIR = "train_images"
TEST_IMG_DIR = "test_images"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------------------------------------
# Вспомогательные функции для загрузки
# --------------------------------------------------
def get_direct_file_link(mailru_file_url: str) -> str:
    resp = requests.get(mailru_file_url)
    if resp.status_code != 200:
        raise RuntimeError(f"Ошибка {resp.status_code} при запросе {mailru_file_url}")
    page = resp.text
    match = re.search(r'dispatcher.*?weblink_get.*?url":"(.*?)"', page)
    if not match:
        raise RuntimeError("Не удалось найти CDN ссылку в HTML")
    base_url = match.group(1)
    parts = mailru_file_url.split('/')[-3:]
    return f"{base_url}/{parts[0]}/{parts[1]}/{parts[2]}"

def download_from_mailru(file_url: str, local_name: str):
    direct = get_direct_file_link(file_url)
    print(f"Скачиваем {file_url} → {local_name}")
    os.system(f"wget --content-disposition '{direct}' -O '{local_name}'")

def build_id_to_paths(img_dir: str) -> dict:
    id_to_files = defaultdict(list)
    for fname in os.listdir(img_dir):
        if fname.endswith(".jpg"):
            iid = int(fname.split("_")[0])
            id_to_files[iid].append(os.path.join(img_dir, fname))
    return id_to_files

# --------------------------------------------------
# Модель для дообучения
# --------------------------------------------------
class FineTunedViT(nn.Module):
    def __init__(self, model_name="vit_tiny_patch16_224", pretrained=True, dropout=0.1):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0
        )
        self.embed_dim = self.backbone.embed_dim

        self.regression_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.embed_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        price_pred = self.regression_head(features)
        return price_pred

    def get_embeddings(self, x):
        with torch.no_grad():
            return self.backbone(x)

class CarPriceDataset(Dataset):
    def __init__(self, id_to_paths, id_to_price, transform, max_images=4):
        self.id_to_paths = id_to_paths
        self.id_to_price = id_to_price
        self.transform = transform
        self.max_images = max_images
        self.ids = list(id_to_price.keys())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        item_id = self.ids[idx]
        paths = self.id_to_paths.get(item_id, [])[:self.max_images]

        if not paths:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
            img = Image.fromarray(img)
            if self.transform:
                img = self.transform(img)
            return img, self.id_to_price[item_id], 0

        random_path = np.random.choice(paths)
        try:
            img = cv2.imread(random_path)
            if img is None:
                img = np.zeros((224, 224, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(img)
        except:
            img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))

        if self.transform:
            img = self.transform(img)

        return img, self.id_to_price[item_id], 1

def download_and_extract_data():
    """Скачиваем и распаковываем данные если нужно"""
    train_link = "https://cloud.mail.ru/public/2kaD/W4xWY9vgr/train_images.zip"
    test_link = "https://cloud.mail.ru/public/2kaD/W4xWY9vgr/test_images.zip"

    if not os.path.exists(TRAIN_IMG_ZIP):
        print("Скачиваем train_images.zip...")
        download_from_mailru(train_link, TRAIN_IMG_ZIP)
    if not os.path.exists(TEST_IMG_ZIP):
        print("Скачиваем test_images.zip...")
        download_from_mailru(test_link, TEST_IMG_ZIP)

    if not os.path.exists(TRAIN_IMG_DIR):
        print("Распаковываем train_images.zip...")
        with zipfile.ZipFile(TRAIN_IMG_ZIP, 'r') as zip_ref:
            zip_ref.extractall(TRAIN_IMG_DIR)
    if not os.path.exists(TEST_IMG_DIR):
        print("Распаковываем test_images.zip...")
        with zipfile.ZipFile(TEST_IMG_ZIP, 'r') as zip_ref:
            zip_ref.extractall(TEST_IMG_DIR)

def train_fine_tuned_model():
    """Дообучение модели на регрессию цен"""
    # Загрузка данных
    #download_and_extract_data()

    train_df = pd.read_parquet(TRAIN_PQ)
    train_id_to_paths = build_id_to_paths(TRAIN_IMG_DIR)

    # Создаем словарь ID -> логарифм цены
    id_to_price = {}
    for _, row in train_df.iterrows():
        price = row['price_TARGET']
        log_price = np.log1p(price)
        id_to_price[row['ID']] = log_price

    # Модель и трансформы
    model = FineTunedViT()
    model = model.to(DEVICE)

    config = resolve_data_config({}, model=model.backbone)
    train_transform = create_transform(**config)

    # Датасет и даталоадер
    dataset = CarPriceDataset(train_id_to_paths, id_to_price, train_transform)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

    # Оптимизатор и функция потерь
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    criterion = nn.MSELoss()

    # Обучение
    model.train()
    best_loss = float('inf')

    for epoch in range(20):
        total_loss = 0
        progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1}/20')

        for batch_imgs, batch_prices, batch_flags in progress_bar:
            batch_imgs = batch_imgs.to(DEVICE)
            batch_prices = batch_prices.float().to(DEVICE).unsqueeze(1)
            batch_flags = batch_flags.to(DEVICE)

            pred_prices = model(batch_imgs)

            valid_mask = batch_flags == 1
            if valid_mask.sum() > 0:
                loss = criterion(pred_prices[valid_mask], batch_prices[valid_mask])
            else:
                continue

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / len(dataloader)
        print(f'Epoch {epoch+1}, Average Loss: {avg_loss:.4f}')

        # Сохраняем лучшую модель
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), 'fine_tuned_vit.pth')
            print(f"Новая лучшая модель сохранена с loss: {best_loss:.4f}")

    print("Дообучение завершено!")
    return model

# Запуск блока 1
if __name__ == "__main__":
    print("=== БЛОК 1: Загрузка данных и дообучение модели ===")
    model = train_fine_tuned_model()

=== БЛОК 1: Загрузка данных и дообучение модели ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Epoch 1/20: 100%|██████████| 2188/2188 [06:38<00:00,  5.49it/s, loss=0.8113]


Epoch 1, Average Loss: 1.2167
Новая лучшая модель сохранена с loss: 1.2167


...

Epoch 18, Average Loss: 0.1947
Новая лучшая модель сохранена с loss: 0.1947


Epoch 19/20: 100%|██████████| 2188/2188 [06:37<00:00,  5.50it/s, loss=0.2110]


Epoch 19, Average Loss: 0.1850
Новая лучшая модель сохранена с loss: 0.1850


Epoch 20/20: 100%|██████████| 2188/2188 [06:32<00:00,  5.58it/s, loss=0.2477]


Epoch 20, Average Loss: 0.1789
Новая лучшая модель сохранена с loss: 0.1789
Дообучение завершено!


In [ ]:
# --------------------------------------------------
# БЛОК 2: Извлечение улучшенных эмбеддингов
# --------------------------------------------------
import os
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from collections import defaultdict

# Настройки (должны совпадать с Блоком 1)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class ImagePathDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)

        if self.transform:
            img = self.transform(img)
        return img

def extract_fine_tuned_embeddings():
    """Извлечение эмбеддингов с дообученной моделью"""

    # Загружаем дообученную модель
    model = FineTunedViT()
    model.load_state_dict(torch.load('fine_tuned_vit.pth', map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    # Загружаем данные
    train_df = pd.read_parquet("train_dataset.parquet")
    test_df = pd.read_parquet("test_dataset.parquet")

    train_id_to_paths = build_id_to_paths("train_images")
    test_id_to_paths = build_id_to_paths("test_images")

    train_ids = train_df["ID"].tolist()
    test_ids = test_df["ID"].tolist()

    # Трансформы для инференса
    config = resolve_data_config({}, model=model.backbone)
    transform = create_transform(**config)

    def extract_embeddings(id_list, id_to_paths, desc="Extracting"):
        all_embeddings = {}

        for item_id in tqdm(id_list, desc=desc):
            paths = id_to_paths.get(item_id, [])
            if not paths:
                emb_size = model.embed_dim
                all_embeddings[item_id] = np.zeros(emb_size, dtype=np.float32)
                continue

            img_dataset = ImagePathDataset(paths, transform=transform)
            img_loader = DataLoader(
                img_dataset,
                batch_size=64,
                shuffle=False,
                num_workers=0,
                pin_memory=True
            )

            embeddings = []
            with torch.no_grad():
                for batch in img_loader:
                    batch = batch.to(DEVICE, non_blocking=True)
                    feats = model.get_embeddings(batch)
                    embeddings.append(feats.cpu().numpy())

            if embeddings:
                mean_emb = np.mean(np.vstack(embeddings), axis=0)
            else:
                emb_size = model.embed_dim
                mean_emb = np.zeros(emb_size, dtype=np.float32)

            all_embeddings[item_id] = mean_emb.astype(np.float32)

        return all_embeddings

    # Извлекаем эмбеддинги
    print("Извлечение улучшенных эмбеддингов для train...")
    train_embeddings = extract_embeddings(train_ids, train_id_to_paths, "Train")

    print("Извлечение улучшенных эмбеддингов для test...")
    test_embeddings = extract_embeddings(test_ids, test_id_to_paths, "Test")

    # Сохраняем
    EMB_SIZE = model.embed_dim

    train_emb_df = pd.DataFrame.from_dict(train_embeddings, orient='index')
    train_emb_df.index.name = 'ID'
    train_emb_df.columns = [f'img_emb_{i}' for i in range(EMB_SIZE)]

    test_emb_df = pd.DataFrame.from_dict(test_embeddings, orient='index')
    test_emb_df.index.name = 'ID'
    test_emb_df.columns = [f'img_emb_{i}' for i in range(EMB_SIZE)]

    train_emb_df.to_parquet("train_fine_tuned_embeddings.parquet")
    test_emb_df.to_parquet("test_fine_tuned_embeddings.parquet")

    print(f"Улучшенные эмбеддинги (размерность {EMB_SIZE}) сохранены!")
    return train_emb_df, test_emb_df

# Запуск блока 2
if __name__ == "__main__":
    print("=== БЛОК 2: Извлечение улучшенных эмбеддингов ===")
    train_emb, test_emb = extract_fine_tuned_embeddings()

=== БЛОК 2: Извлечение улучшенных эмбеддингов ===
Извлечение улучшенных эмбеддингов для train...


Train: 100%|██████████| 70000/70000 [36:29<00:00, 31.96it/s]


Извлечение улучшенных эмбеддингов для test...


Test: 100%|██████████| 25000/25000 [12:10<00:00, 34.24it/s]


Улучшенные эмбеддинги (размерность 192) сохранены!


In [ ]:
!pip install catboost > _

In [ ]:
# --------------------------------------------------
# БЛОК 3: Обучение мощного CatBoost на улучшенных эмбеддингах
# --------------------------------------------------
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split

# Ваши функции обработки (оставляем как есть)
def process_multichoice_columns(df):
    """Преобразует мультивыборные поля в числовые (количество опций)."""
    df = df.copy()
    mult_cols = [
        'aktivnaya_bezopasnost_mult', 'audiosistema_mult', 'shini_i_diski_mult',
        'electroprivod_mult', 'fary_mult', 'multimedia_navigacia_mult',
        'obogrev_mult', 'pamyat_nastroek_mult', 'podushki_bezopasnosti_mult',
        'pomosh_pri_vozhdenii_mult', 'protivoygonnaya_sistema_mult',
        'salon_mult', 'upravlenie_klimatom_mult'
    ]

    for col in mult_cols:
        if col in df.columns:
            def count_options(x):
                if x is None:
                    return 0
                try:
                    if pd.isna(x):
                        return 0
                except:
                    pass

                if isinstance(x, (list, np.ndarray)):
                    if len(x) == 0:
                        return 0
                    if len(x) == 1 and x[0] is None:
                        return 0
                    valid_count = sum(1 for item in x if item is not None)
                    return valid_count

                return 0

            df[f'{col}_count'] = df[col].apply(count_options)
            if col in df.columns:
                df = df.drop(columns=[col])
    return df

# Основной код обучения

# Загружаем ВСЕ данные с улучшенными эмбеддингами
train_df = pd.read_parquet("train_dataset.parquet")
test_df = pd.read_parquet("test_dataset.parquet")

# ИСПОЛЬЗУЕМ УЛУЧШЕННЫЕ ЭМБЕДДИНГИ!
train_emb = pd.read_parquet("train_fine_tuned_embeddings.parquet")
test_emb = pd.read_parquet("test_fine_tuned_embeddings.parquet")

# Обработка обоих датасетов (ваш текущий код)
train_processed = train_df.copy()
test_processed = test_df.copy()

# Логарифмируем пробег
train_processed['mileage_log'] = np.log1p(train_processed['mileage'])
test_processed['mileage_log'] = np.log1p(test_processed['mileage'])
train_processed = train_processed.drop(columns=['mileage'])
test_processed = test_processed.drop(columns=['mileage'])

# Обработка мультивыборных полей
print("Обработка мультивыборных полей...")
train_processed = process_multichoice_columns(train_processed)
test_processed = process_multichoice_columns(test_processed)

# Объединение с УЛУЧШЕННЫМИ эмбеддингами
train_final = train_processed.set_index("ID").join(train_emb, how='left')
test_final = test_processed.set_index("ID").join(test_emb, how='left')

# Логарифмируем целевую переменную
y_train_log = np.log1p(train_final['price_TARGET'])
X_train = train_final.drop(columns=['price_TARGET'])
X_test = test_final.copy()

# Подготовка категориальных признаков (ваш текущий код)
categorical_features = [
    'equipment', 'body_type', 'drive_type', 'engine_type', 'doors_number',
    'color', 'pts', 'steering_wheel', 'audiosistema', 'diski',
    'electropodemniki', 'fary', 'salon', 'upravlenie_klimatom', 'usilitel_rul'
]

# Автоматическое обнаружение строковых столбцов
object_columns = X_train.select_dtypes(include=['object']).columns.tolist()
missing_categorical = [col for col in object_columns if col not in categorical_features]

if missing_categorical:
    print("Обнаружены строковые столбцы, не указанные как категориальные:", missing_categorical)
    categorical_features.extend(missing_categorical)
    print("Обновленный список категориальных признаков:", categorical_features)

# Преобразование не-категориальных столбцов в числовые
non_cat_columns = [col for col in X_train.columns if col not in categorical_features]
for col in non_cat_columns:
    if X_train[col].dtype == 'object':
        X_train[col] = pd.to_numeric(X_train[col].astype(str).str.replace(r'[^\d.-]', '', regex=True), errors='coerce')
        X_test[col] = pd.to_numeric(X_test[col].astype(str).str.replace(r'[^\d.-]', '', regex=True), errors='coerce')
    X_train[col] = X_train[col].fillna(0)
    X_test[col] = X_test[col].fillna(0)

# Финальная обработка категориальных признаков
for col in categorical_features:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('missing')
        X_test[col] = X_test[col].fillna('missing')
        X_train[col] = X_train[col].astype('str')
        X_test[col] = X_test[col].astype('str')

# Заполняем пропуски в числовых колонках
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
X_train[numeric_cols] = X_train[numeric_cols].fillna(0)
X_test[numeric_cols] = X_test[numeric_cols].fillna(0)

print("Финальные формы данных:")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# Обучение CatBoost (ваш текущий код с GPU и RMSE)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_log, test_size=0.2, random_state=42, shuffle=True
)

train_pool = Pool(X_tr, y_tr, cat_features=categorical_features)
valid_pool = Pool(X_val, y_val, cat_features=categorical_features)

print("Обучение CatBoostRegressor на GPU с RMSE...")
model = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.05,
    depth=10,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    task_type='GPU',
    od_type='Iter',
    od_wait=100,
    verbose=100
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# Предсказание
test_pool = Pool(X_test, cat_features=categorical_features)
y_pred_log = model.predict(test_pool)
y_pred = np.expm1(y_pred_log)

submission = pd.DataFrame({
    'ID': X_test.index,
    'target': y_pred
})
submission.to_csv("submission_fine_tuned_embeddings.csv", index=False)
print("Готово! Файл submission_fine_tuned_embeddings.csv создан.")
print(f"Размер submission: {submission.shape}")

Обработка мультивыборных полей...
Обнаружены строковые столбцы, не указанные как категориальные: ['crashes_count', 'owners_count']
Обновленный список категориальных признаков: ['equipment', 'body_type', 'drive_type', 'engine_type', 'doors_number', 'color', 'pts', 'steering_wheel', 'audiosistema', 'diski', 'electropodemniki', 'fary', 'salon', 'upravlenie_klimatom', 'usilitel_rul', 'crashes_count', 'owners_count']
Финальные формы данных:
X_train: (70000, 225), X_test: (25000, 225)
Обучение CatBoostRegressor на GPU с RMSE...
0:	learn: 0.9645511	test: 0.9475400	best: 0.9475400 (0)	total: 469ms	remaining: 31m 15s
100:	learn: 0.2495976	test: 0.2574911	best: 0.2574911 (100)	total: 12.7s	remaining: 8m 11s
200:	learn: 0.2355056	test: 0.2526883	best: 0.2526883 (200)	total: 21.5s	remaining: 6m 46s
300:	learn: 0.2235007	test: 0.2505061	best: 0.2505061 (300)	total: 28.4s	remaining: 5m 48s
400:	learn: 0.2122680	test: 0.2492316	best: 0.2492316 (400)	total: 37.2s	remaining: 5m 33s
500:	learn: 0.203221

In [ ]:
print("Обучение CatBoostRegressor на GPU с RMSE...")
model = CatBoostRegressor(
    iterations=10000,
    learning_rate=0.01,
    depth=8,
    loss_function='RMSE',
    eval_metric='RMSE',
    task_type='GPU',
    od_type='Iter',
    od_wait=200,
    verbose=100
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# Предсказание
test_pool = Pool(X_test, cat_features=categorical_features)
y_pred_log = model.predict(test_pool)
y_pred = np.expm1(y_pred_log)

submission = pd.DataFrame({
    'ID': X_test.index,
    'target': y_pred
})
submission.to_csv("submission_fine_tuned_embeddings.csv", index=False)
print("Готово! Файл submission_fine_tuned_embeddings.csv создан.")
print(f"Размер submission: {submission.shape}")

Обучение CatBoostRegressor на GPU с RMSE...
0:	learn: 0.9999966	test: 0.9822547	best: 0.9822547 (0)	total: 57.9ms	remaining: 9m 38s
100:	learn: 0.4616709	test: 0.4537385	best: 0.4537385 (100)	total: 6.25s	remaining: 10m 12s
200:	learn: 0.3062497	test: 0.3043477	best: 0.3043477 (200)	total: 12.4s	remaining: 10m 4s
300:	learn: 0.2694286	test: 0.2707212	best: 0.2707212 (300)	total: 16.3s	remaining: 8m 44s
400:	learn: 0.2594938	test: 0.2624906	best: 0.2624906 (400)	total: 20.2s	remaining: 8m 4s
500:	learn: 0.2553668	test: 0.2594649	best: 0.2594649 (500)	total: 26.6s	remaining: 8m 24s
600:	learn: 0.2527305	test: 0.2577436	best: 0.2577436 (600)	total: 30.5s	remaining: 7m 57s
700:	learn: 0.2505909	test: 0.2564897	best: 0.2564897 (700)	total: 36.7s	remaining: 8m 7s
800:	learn: 0.2486982	test: 0.2554330	best: 0.2554330 (800)	total: 41.8s	remaining: 8m
900:	learn: 0.2470409	test: 0.2546113	best: 0.2546113 (900)	total: 46.6s	remaining: 7m 50s
1000:	learn: 0.2454632	test: 0.2538805	best: 0.2538805